🧠 Big picture (what this task is)
training a model to read a sentence and decide if it's positive or negative.

Practical 1: Sentiment Analysis – Simple ANN &
Transformers

**Task 1.1**

Preprocessing steps in data_loading_code.py provided

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from matplotlib import pyplot
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk import word_tokenize
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, classification_report
from google.colab import files
import io
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


def preprocess_pandas(data, columns):
    df_ = pd.DataFrame(columns=columns)

    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace(r'[a-zA-Z0-9-_.]+@[a-zA-Z0-9-_.]+', '', regex=True)
    data['Sentence'] = data['Sentence'].replace(r'((25[0-5]|2[0-4][0-9]|[01]?[0-9][0-9]?)(\.|$)){4}', '', regex=True)
    data['Sentence'] = data['Sentence'].str.replace(r'[^\w\s]', '', regex=True)
    data['Sentence'] = data['Sentence'].replace(r'\d', '', regex=True)

    for index, row in data.iterrows():
        word_tokens = word_tokenize(row['Sentence'])
        filtered_sent = [w for w in word_tokens if w not in stopwords.words('english')]

        df_.loc[len(df_)] = {
            "index": row['index'],
            "Class": row['Class'],
            "Sentence": " ".join(filtered_sent)
        }

    return data


if __name__ == "__main__":

    # Load data
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]

    data = pd.read_csv(io.BytesIO(uploaded[filename]), delimiter='\t', header=None)
    data.columns = ['Sentence', 'Class']
    data['index'] = data.index

    columns = ['index', 'Class', 'Sentence']
    data = preprocess_pandas(data, columns)

    # Split data
    X = data['Sentence'].values.astype('U')
    y = data['Class'].values.astype('int32')

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.1,
        random_state=0,
        shuffle=True
    )

    # TF-IDF vectorization
    word_vectorizer = TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 2),
        max_features=50000,
        max_df=0.5,
        use_idf=True,
        norm='l2'
    )

    # FIT ONLY ON TRAIN
    X_train_tfidf = word_vectorizer.fit_transform(X_train)
    X_val_tfidf = word_vectorizer.transform(X_val)

    # convert sparse → dense
    X_train_tfidf = X_train_tfidf.todense()
    X_val_tfidf = X_val_tfidf.todense()

    # Convert to tensors
    train_x_tensor = torch.from_numpy(np.array(X_train_tfidf)).float()
    train_y_tensor = torch.from_numpy(np.array(y_train)).long()

    validation_x_tensor = torch.from_numpy(np.array(X_val_tfidf)).float()
    validation_y_tensor = torch.from_numpy(np.array(y_val)).long()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saving amazon_cells_labelled.txt to amazon_cells_labelled (8).txt


Model 1: Simple ANN

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleANN(nn.Module):
    def __init__(self, input_size):
        super(SimpleANN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 2)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
model = SimpleANN(train_x_tensor.shape[1])

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 30

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    outputs = model(train_x_tensor)
    loss = criterion(outputs, train_y_tensor)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/30, Loss: 0.6938
Epoch 2/30, Loss: 0.6921
Epoch 3/30, Loss: 0.6902
Epoch 4/30, Loss: 0.6878
Epoch 5/30, Loss: 0.6850
Epoch 6/30, Loss: 0.6818
Epoch 7/30, Loss: 0.6778
Epoch 8/30, Loss: 0.6734
Epoch 9/30, Loss: 0.6681
Epoch 10/30, Loss: 0.6618
Epoch 11/30, Loss: 0.6552
Epoch 12/30, Loss: 0.6477
Epoch 13/30, Loss: 0.6390
Epoch 14/30, Loss: 0.6302
Epoch 15/30, Loss: 0.6198
Epoch 16/30, Loss: 0.6083
Epoch 17/30, Loss: 0.5963
Epoch 18/30, Loss: 0.5832
Epoch 19/30, Loss: 0.5693
Epoch 20/30, Loss: 0.5542
Epoch 21/30, Loss: 0.5400
Epoch 22/30, Loss: 0.5231
Epoch 23/30, Loss: 0.5057
Epoch 24/30, Loss: 0.4873
Epoch 25/30, Loss: 0.4685
Epoch 26/30, Loss: 0.4488
Epoch 27/30, Loss: 0.4288
Epoch 28/30, Loss: 0.4066
Epoch 29/30, Loss: 0.3848
Epoch 30/30, Loss: 0.3624


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

model.eval()

with torch.no_grad():
    outputs = model(validation_x_tensor)
    preds = torch.argmax(outputs, dim=1)

print("Accuracy:", accuracy_score(validation_y_tensor, preds))
print("Precision:", precision_score(validation_y_tensor, preds))
print("Recall:", recall_score(validation_y_tensor, preds))

Accuracy: 0.83
Precision: 0.8214285714285714
Recall: 0.8679245283018868


Model 2: LSTM
For LSTM we must use word sequences, TF-IDF is not usable for LSTM

In [ ]:
# Tokenize text
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

In [ ]:
# Convert text to sequences

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)

In [ ]:
# padding, make them into same length
max_len = 100

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='pre')
X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding='pre')

In [ ]:
# convert to tensors
y_train_tensor = torch.tensor(y_train).long()
y_val_tensor = torch.tensor(y_val).long()

X_train_tensor = torch.tensor(X_train_pad).long()
X_val_tensor = torch.tensor(X_val_pad).long()

In [ ]:
# Create dataloader
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
import torch.nn as nn

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size=20000, embed_dim=128, hidden_dim=128, output_dim=2):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return out

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMClassifier().to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
model.train()

for epoch in range(5):
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        print(f"Epoch {epoch+1}, avg loss: {total_loss / len(train_loader):.4f}")

Epoch 1, avg loss: 0.0238
Epoch 1, avg loss: 0.0477
Epoch 1, avg loss: 0.0720
Epoch 1, avg loss: 0.0954
Epoch 1, avg loss: 0.1188
Epoch 1, avg loss: 0.1419
Epoch 1, avg loss: 0.1644
Epoch 1, avg loss: 0.1880
Epoch 1, avg loss: 0.2097
Epoch 1, avg loss: 0.2334
Epoch 1, avg loss: 0.2566
Epoch 1, avg loss: 0.2800
Epoch 1, avg loss: 0.3032
Epoch 1, avg loss: 0.3266
Epoch 1, avg loss: 0.3489
Epoch 1, avg loss: 0.3708
Epoch 1, avg loss: 0.3938
Epoch 1, avg loss: 0.4165
Epoch 1, avg loss: 0.4381
Epoch 1, avg loss: 0.4605
Epoch 1, avg loss: 0.4832
Epoch 1, avg loss: 0.5043
Epoch 1, avg loss: 0.5264
Epoch 1, avg loss: 0.5479
Epoch 1, avg loss: 0.5705
Epoch 1, avg loss: 0.5924
Epoch 1, avg loss: 0.6145
Epoch 1, avg loss: 0.6371
Epoch 1, avg loss: 0.6597
Epoch 2, avg loss: 0.0183
Epoch 2, avg loss: 0.0401
Epoch 2, avg loss: 0.0597
Epoch 2, avg loss: 0.0805
Epoch 2, avg loss: 0.1003
Epoch 2, avg loss: 0.1193
Epoch 2, avg loss: 0.1375
Epoch 2, avg loss: 0.1548
Epoch 2, avg loss: 0.1720
Epoch 2, avg

In [ ]:
print("sample sequence:", X_train_pad[0])
print("label distribution:", np.bincount(y_train))
print("non-zero ratio:", (X_train_pad > 0).mean())

sample sequence: [  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0
   0   0   0   0   0   0   0   0   0   0   0   0   8 498   2 746  14 747
 748  12   2 125 499   4  51  52   2  91]
label distribution: [453 447]
non-zero ratio: 0.10264444444444444


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

model.eval()
preds = []
true = []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch)
        predicted = torch.argmax(outputs, dim=1).cpu().numpy()

        preds.extend(predicted)
        true.extend(y_batch.numpy())

print("Accuracy:", accuracy_score(true, preds))
print("Precision:", precision_score(true, preds))
print("Recall:", recall_score(true, preds))

Accuracy: 0.82
Precision: 0.8571428571428571
Recall: 0.7924528301886793


The Improvement Process
Baseline Performance: Initially, the LSTM model suffered from mode collapse, yielding an accuracy of 0.53 and a recall of 1.0, indicating it was merely predicting the majority class.

Architectural/Preprocessing Fixes: By identifying that the model was failing to capture the sentiment signal, you improved the pipeline—likely by switching to pre-padding or adjusting the LSTM architecture—to ensure the network could maintain long-term dependencies from the text.

Final Outcome: These interventions resulted in a functional model with balanced metrics (Accuracy: 0.82, Precision: 0.86, Recall: 0.79), which is a 55% relative improvement over the baseline's predictive power.

**Task 1.2**

Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
import io
import math
from google.colab import files
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.metrics import accuracy_score, precision_score, recall_score
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

def preprocess_pandas(data):
    data['Sentence'] = data['Sentence'].str.lower()
    data['Sentence'] = data['Sentence'].replace(r'[^\w\s]', '', regex=True)
    data['Sentence'] = data['Sentence'].replace(r'\d', '', regex=True)
    stop_words = set(stopwords.words('english'))
    processed_sentences = []
    for sent in data['Sentence']:
        word_tokens = word_tokenize(sent)
        filtered = [w for w in word_tokens if w not in stop_words]
        processed_sentences.append(" ".join(filtered))
    data['Sentence'] = processed_sentences
    return data

uploaded = files.upload()
filename = list(uploaded.keys())[0]

data = pd.read_csv(io.BytesIO(uploaded[filename]), delimiter='\t', header=None, engine='python')
data.columns = ['Sentence', 'Class']
data = preprocess_pandas(data)

X = data['Sentence'].values.astype('U')
y = data['Class'].values.astype('int32')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=0)

tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=100, padding='pre')
X_val_pad = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=100, padding='pre')

X_train_tensor = torch.tensor(X_train_pad).long()
y_train_tensor = torch.tensor(y_train).long()
X_val_tensor = torch.tensor(X_val_pad).long()
y_val_tensor = torch.tensor(y_val).long()

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=32)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size=20000, embed_dim=128, nhead=8, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_encoder = PositionalEncoding(embed_dim)
        encoder_layers = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=nhead, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, 2)
    def forward(self, x):
        x = self.pos_encoder(self.embedding(x) * math.sqrt(128))
        x = self.transformer_encoder(x).mean(dim=1)
        return self.fc(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerClassifier().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        criterion(model(xb), yb).backward()
        optimizer.step()

model.eval()
preds, true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        preds.extend(torch.argmax(model(xb.to(device)), dim=1).cpu().numpy())
        true.extend(yb.numpy())

print(f"Accuracy: {accuracy_score(true, preds):.4f}")
print(f"Precision: {precision_score(true, preds):.4f}")
print(f"Recall: {recall_score(true, preds):.4f}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Saving amazon_cells_labelled.txt to amazon_cells_labelled (1).txt
Accuracy: 0.7000
Precision: 0.8108
Recall: 0.5660


After the ANN and LSTM, a Transformer model was implemented to test self-attention on text . Unlike LSTMs, Transformers process sequences in parallel rather than step-by-step.

To handle word order, Positional Encoding was added to the embeddings. A multi-head Transformer Encoder was used for feature extraction, followed by Global Average Pooling to flatten the output for the final classification layer.

The model achieved 0.70 Accuracy, 0.81 Precision, and 0.57 Recall. The lower performance compared to the ANN (0.83) suggests the Transformer's complexity was ill-suited for this small dataset, leading to poor generalization .

**Task 1.3**



The Simple ANN performed best overall with 0.83 accuracy, followed closely by the LSTM at 0.82, and the Transformer last at 0.70.

Simple ANN is preferred when the dataset is small and the task is straightforward and knowing which words appear in a review is enough to classify it correctly.

LSTM is preferred when word order matters, for example when negation or sentence structure changes the meaning. It would also likely outperform the ANN on larger datasets.

Transformer is preferred when we have a large dataset or access to pre-trained models like BERT. On small datasets like this one, it is not the right choice.

Looking at the three models' complexity, accuracy, and efficiency difference, the Simple ANN is the least complex with only 3 linear layers. It trains very fast and gave the best accuracy. It works well here because TF-IDF already captures the most useful information, so the model doesn't need to be complex.

The LSTM is more complex as it processes words one by one in sequence, which takes longer to train. It achieved similar accuracy to the ANN but had the best precision (0.86), meaning when it predicted positive, it was most often right. It did better than the ANN in capturing some sequential patterns in the text.

The transformer is the most complex as it uses multi-head attention and positional encoding, and has the most parameters. Despite this, it performed worst. The reason is that its complexity is a disadvantage on small datasets — it has too many things to learn and not enough examples to learn from. This is reflected in its low recall (0.57), meaning it missed a lot of positive reviews.

Data amount: The more complex the model, the more data it needs. The Transformer clearly did not do well from the small dataset size. The ANN worked well precisely because it is simple enough to learn from limited data. If we had a much larger dataset, the Transformer would likely outperform the others.

Embeddings: The ANN used TF-IDF, which is a simple word-counting method with no understanding of meaning. The LSTM and Transformer used learned embeddings trained from scratch, which should be more powerful, but because the dataset was small, these embeddings didn't have enough data to become meaningful. Using pre-trained embeddings like GloVe or BERT would likely have improved both the LSTM and Transformer results significantly.

Architectural choices: The LSTM initially suffered from mode collapse (accuracy 0.53, recall 1.0), meaning it predicted everything as one class. Switching to pre-padding fixed this and brought accuracy up to 0.82 which shows that small implementation details can have a big impact. For the Transformer, using global average pooling over the encoder output was a reasonable design choice, but the overall architecture was too large for the dataset size, leading to poor generalisation.